In [ ]:
!pip install sleap-io

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 975.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.9/137.9 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 18.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.8/117.8 kB 501.6 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.7/526.7 kB 32.2 MB/s eta 0:00:00


In [ ]:
!gdown --fuzzy https://drive.google.com/file/d/15iKXpcogB217kMXBKeqpcYXqVzGO4wdB/view?usp=sharing
!gdown --fuzzy https://drive.google.com/file/d/1k6SgvG_G41lhRRCK_LHaGEKLPBGS7fBQ/view?usp=drive_link

Downloading...
From (original): https://drive.google.com/uc?id=15iKXpcogB217kMXBKeqpcYXqVzGO4wdB
From (redirected): https://drive.google.com/uc?id=15iKXpcogB217kMXBKeqpcYXqVzGO4wdB&confirm=t&uuid=04cfd2c8-f81e-4ba6-88d9-19f048d389d4
To: /content/top-03132020125012-0000.mp4_1.slp
100% 32.0M/32.0M [00:00<00:00, 121MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1k6SgvG_G41lhRRCK_LHaGEKLPBGS7fBQ
To: /content/without_gaps.npy
100% 2.52M/2.52M [00:00<00:00, 54.0MB/s]


In [ ]:
import numpy as np
import sleap_io as sio
from scipy.optimize import linear_sum_assignment

In [ ]:
labels = sio.load_file("top-03132020125012-0000.mp4_1.slp")
labels

Labels(labeled_frames=52393, videos=1, skeletons=1, tracks=9)

In [ ]:
# Remove original tracks
for lf in labels:
    for inst in lf:
        inst.track = None
labels.tracks = []
labels

Labels(labeled_frames=52393, videos=1, skeletons=1, tracks=0)

In [ ]:
idt = np.load("without_gaps.npy", allow_pickle=True)
idt = idt[()]
idt

{'trajectories': array([[[         nan,          nan],
         [         nan,          nan]],
 
        [[902.3781965 , 387.54912517],
         [         nan,          nan]],
 
        [[908.45450435, 386.46998983],
         [         nan,          nan]],
 
        ...,
 
        [[562.99513437, 714.13227038],
         [         nan,          nan]],
 
        [[561.62233206, 713.61222197],
         [         nan,          nan]],
 
        [[562.02744577, 713.82905415],
         [         nan,          nan]]]),
 'version': '5.2.11',
 'video_paths': ['/Users/soline/Documents/idtrackerai/demo/top-03132020125012-0000.mp4'],
 'frames_per_second': 30,
 'body_length': 366.0,
 'stats': {'estimated_accuracy': 0.9918651663500414,
  'estimated_accuracy_after_interpolation': 0.9973305904537996,
  'percentage_identified': 0.3909587158589888,
  'estimated_accuracy_identified': 0.9973305904537996},
 'areas': {'mean': array([3932.50205202, 3597.3280052 ]),
  'median': array([3955.5, 3578.5]),
  'std'

In [ ]:
idt_trx = idt["trajectories"]
idt_trx.shape  # (frames, tracks, xy)

(52393, 2, 2)

In [ ]:
# Create SLEAP tracks for each idtracker track.
n_idt_tracks = idt_trx.shape[1]
idt_tracks = [sio.Track(name=f"idtracker_{i}")for i in range(n_idt_tracks)]

# Save to labels.
labels.tracks = idt_tracks

In [ ]:
for lf in labels:

    if len(lf) == 0:
        continue

    # Get SLEAP centroids.
    pts = lf.numpy()  # (instances, nodes, xy)
    slp_centroids = np.nanmedian(pts, axis=1)  # (instances, xy)

    # Get idtracker centroids.
    idt_centroids = idt_trx[lf.frame_idx]  # (tracks, xy)
    not_missing = ~(np.isnan(idt_centroids).any(axis=-1))
    idt_track_inds = np.argwhere(not_missing).reshape(-1)  # (n_tracks,)
    idt_centroids = idt_centroids[not_missing]

    if len(idt_centroids) == 0:
        continue

    # Match centroids.
    cost = np.linalg.norm(
        np.expand_dims(idt_centroids, axis=1) -
        np.expand_dims(slp_centroids, axis=0),
        axis=-1
    )
    matches_idt, matches_slp = linear_sum_assignment(cost)

    # Assign idtracker tracks.
    for idt_ind, slp_ind in zip(matches_idt, matches_slp):
        lf[slp_ind].track = idt_tracks[idt_track_inds[idt_ind]]


In [ ]:
labels.save("idtracked.slp")